In [15]:
# Remove unwanted warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', catgeory=RuntimeWarning)

import os

# Data Management
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
from pandas_datareader.data import DataReader
from ta import add_all_ta_features

# Statistics
from statsmodels.tsa.stattools import adfuller

# Unsupervised Machine Learning
from sklearn.decomposition import PCA

# Supervised Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

# Reporting
import matplotlib.pyplot as plt

In [3]:
# Global Variables
CSV_FILENAME = "stocks.csv"
PARQET_FILENAME = "stocks.parquet"
WORKING_DIR = "/home/mahan-maalekian/Downloads/"
FEATURES = ["gvkey"]

### Data Extraction

In [12]:
if not os.path.exists(os.path.join(
        WORKING_DIR, "ret_sample.csv"
    )):
    # read sample data
    file_path = os.path.join(
        WORKING_DIR, "ret_sample.csv"
    )
    raw = pl.read_csv(file_path)
    raw = raw.filter(pl.col("excntry").is_in(["CAN","USA"]))
    raw.write_csv(CSV_FILENAME)

if not os.path.exists(PARQET_FILENAME):
    raw = pd.read_csv(CSV_FILENAME, dtype={4: str})
    raw.to_parquet(PARQET_FILENAME, index=False, compression="snappy")

raw = pd.read_parquet(PARQET_FILENAME)

In [14]:
raw = raw[["id"]]
raw.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1398807 entries, 0 to 1398806
Data columns (total 1 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   id      1398807 non-null  object
dtypes: object(1)
memory usage: 90.9 MB


statistic,id,date,ret_eom,gvkey,iid,excntry,stock_ret,year,month,char_date,char_eom,me,prc,market_equity,div12m_me,chcsho_12m,eqnpo_12m,ret_1_0,ret_3_1,ret_6_1,ret_9_1,ret_12_1,ret_12_7,ret_60_12,seas_1_1an,seas_1_1na,seas_2_5an,seas_2_5na,at_gr1,sale_gr1,capx_gr1,inv_gr1,debt_gr3,sale_gr3,capx_gr3,inv_gr1a,…,beta_60m,resff3_12_1,resff3_6_1,mispricing_mgmt,mispricing_perf,zero_trades_21d,dolvol_126d,dolvol_var_126d,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_252d,bidaskhl_21d,ivol_capm_21d,iskew_capm_21d,coskew_21d,beta_dimson_21d,ivol_ff3_21d,iskew_ff3_21d,ivol_hxz4_21d,iskew_hxz4_21d,rmax5_21d,rmax1_21d,rvol_21d,rskew_21d,ami_126d,ivol_capm_252d,betadown_252d,prc_highprc_252d,corr_1260d,betabab_1260d,rmax5_rvol_21d,age,qmj,qmj_prof,qmj_growth,qmj_safety
str,str,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""1398807""",1.398807e6,1.398807e6,1.398356e6,"""1398356""","""1398807""",1.398807e6,1.398807e6,1.398807e6,1.398807e6,1.398807e6,1.39721e6,1.398607e6,1.39721e6,1.142563e6,1.308e6,1.304976e6,1.339178e6,1.3238e6,1.30127e6,1.27913e6,1.257266e6,1.257474e6,982934.0,1.284056e6,1.147923e6,958485.0,723172.0,1.272592e6,1.174396e6,1.182394e6,852062.0,962242.0,1.069143e6,1.055586e6,1.257024e6,…,1.124507e6,1.195634e6,1.194735e6,1.268837e6,1.353624e6,1.390683e6,1.375864e6,1.375059e6,1.374494e6,1.373747e6,1.37472e6,1.352983e6,1.390998e6,1.312965e6,1.312965e6,1.312965e6,1.312965e6,1.298549e6,1.298549e6,1.298549e6,1.298549e6,1.312965e6,1.312965e6,1.312965e6,1.312965e6,1.306696e6,1.285074e6,1.275328e6,1.283652e6,1.077072e6,1.066289e6,1.266731e6,1.398807e6,1.016518e6,1.283182e6,1.016531e6,1.316697e6
"""null_count""","""0""",0.0,0.0,451.0,"""451""","""0""",0.0,0.0,0.0,0.0,0.0,1597.0,200.0,1597.0,256244.0,90807.0,93831.0,59629.0,75007.0,97537.0,119677.0,141541.0,141333.0,415873.0,114751.0,250884.0,440322.0,675635.0,126215.0,224411.0,216413.0,546745.0,436565.0,329664.0,343221.0,141783.0,…,274300.0,203173.0,204072.0,129970.0,45183.0,8124.0,22943.0,23748.0,24313.0,25060.0,24087.0,45824.0,7809.0,85842.0,85842.0,85842.0,85842.0,100258.0,100258.0,100258.0,100258.0,85842.0,85842.0,85842.0,85842.0,92111.0,113733.0,123479.0,115155.0,321735.0,332518.0,132076.0,0.0,382289.0,115625.0,382276.0,82110.0
"""mean""",null,2.0143e7,2.0143e7,76895.547734,null,null,0.021684,2014.26942,6.48398,2.0143e7,2.0143e7,5090.292838,32.647701,5090.292838,0.022931,2.292742,-0.070948,0.013901,0.022774,0.055795,0.081137,0.114633,0.050682,0.722298,0.010888,0.009009,0.014988,0.013074,120.448635,1.273326,2.9180e12,0.926002,27.093114,5.894325,1.2727e13,0.002676,…,1.223368,-0.034044,-0.074329,0.500156,0.493877,1.059058,3.1418e7,1.365882,0.009866,1.341279,1.066319,1.0748,0.018441,0.028793,0.207794,-0.005685,1.047222,0.027862,0.174073,0.027637,0.160044,0.040847,0.073339,0.032303,0.192201,5.614744,0.031363,0.988221,0.728693,0.403967,1.113243,1.194626,250.722436,0.001984,0.001482,0.000985,0.001919
"""std""",null,59670.778296,59670.795189,69274.470564,null,null,9.299451,5.96787,3.433929,59674.238985,59674.266059,36161.752443,691.829114,36161.752443,0.392985,975.830437,0.343712,4.770338,5.053951,7.850044,0.947957,1.54096,0.883235,3.319082,0.498165,0.058539,0.216873,0.021626,35450.95285,113.779529,1.0537e15,57.88242,892.214564,459.780425,7.2570e15,0.177155,…,0.777168,0.339754,1.249594,0.16667,0.215733,3.349098,1.0730e8,1.215391,0.079256,1.174741,3.220575,3.189499,0.040296,0.024019,0.987823,0.313852,2.125388,0.023836,0.898926,0.023793,0.849898,0.034762,0.071408,0.025082,0.962465,90.733763,0.020667,0.713607,0.236696,0.187942,0.587813,0.536031,202.100878,0.999177,0.999703,0.999294,0.998989
"""min""","""comp_001004_01""",2.0050201e7,2.0050228e7,1004.0,"""01""","""CAN""",-0.9999,2005.0,1.0,2.005012e7,2.0050131e7,0.0,0.000098

In [10]:
raw.columns

['id',
 'date',
 'ret_eom',
 'gvkey',
 'iid',
 'excntry',
 'stock_ret',
 'year',
 'month',
 'char_date',
 'char_eom',
 'me',
 'prc',
 'market_equity',
 'div12m_me',
 'chcsho_12m',
 'eqnpo_12m',
 'ret_1_0',
 'ret_3_1',
 'ret_6_1',
 'ret_9_1',
 'ret_12_1',
 'ret_12_7',
 'ret_60_12',
 'seas_1_1an',
 'seas_1_1na',
 'seas_2_5an',
 'seas_2_5na',
 'at_gr1',
 'sale_gr1',
 'capx_gr1',
 'inv_gr1',
 'debt_gr3',
 'sale_gr3',
 'capx_gr3',
 'inv_gr1a',
 'lti_gr1a',
 'sti_gr1a',
 'coa_gr1a',
 'col_gr1a',
 'cowc_gr1a',
 'ncoa_gr1a',
 'ncol_gr1a',
 'nncoa_gr1a',
 'fnl_gr1a',
 'nfna_gr1a',
 'tax_gr1a',
 'be_gr1a',
 'ebit_sale',
 'gp_at',
 'cop_at',
 'ope_be',
 'ni_be',
 'ebit_bev',
 'netis_at',
 'eqnetis_at',
 'dbnetis_at',
 'oaccruals_at',
 'oaccruals_ni',
 'taccruals_at',
 'taccruals_ni',
 'noa_at',
 'opex_at',
 'at_turnover',
 'sale_bev',
 'rd_sale',
 'cash_at',
 'sale_emp_gr1',
 'emp_gr1',
 'ni_inc8q',
 'noa_gr1a',
 'ppeinv_gr1a',
 'lnoa_gr1a',
 'capx_gr2',
 'saleq_gr1',
 'niq_be',
 'niq_at',
 'niq_